# CSV statistics

Profiles of each file in the synthetic microbiology dataset. Every count below is computed from the CSVs in this folder. The records are synthetic.

A culture has zero to two isolates, and each isolate has susceptibility results. Keys are `patient_id`, `culture_id`, and `isolate_id`.


## Setup

Loads the six CSV files with pandas. From this folder:

```bash
uv run --with pandas jupyter lab csv_statistics.ipynb
```


In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd()
if not (DATA_DIR / "patients.csv").exists():
    raise FileNotFoundError("Run this notebook from the dataset directory.")

FILES = [
    "patients.csv",
    "wards.csv",
    "antibiotics.csv",
    "cultures.csv",
    "isolates.csv",
    "susceptibility.csv",
]
ID_COLUMNS = {"patient_id", "culture_id", "isolate_id"}
DATE_COLUMNS = {"sample_date", "admission_date"}

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 40)


def load_tables(directory: Path) -> dict[str, pd.DataFrame]:
    """Read every CSV and parse culture dates."""
    frames = {name: pd.read_csv(directory / name) for name in FILES}

    cultures = frames["cultures.csv"]
    cultures["sample_date"] = pd.to_datetime(cultures["sample_date"])
    cultures["admission_date"] = pd.to_datetime(cultures["admission_date"])

    return frames


def missingness(frame: pd.DataFrame) -> pd.DataFrame:
    """Return dtype, missing counts, and distinct-value counts per column."""
    return pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "non_null": frame.notna().sum(),
            "missing": frame.isna().sum(),
            "missing_pct": (frame.isna().mean() * 100).round(2),
            "nunique": frame.nunique(dropna=True),
        }
    )


def numeric_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Describe numeric columns."""
    numeric = frame.select_dtypes(include="number")
    if numeric.empty:
        return numeric

    described = numeric.describe().T
    described["missing"] = numeric.isna().sum()
    return described


def category_counts(series: pd.Series, limit: int = 20) -> pd.DataFrame:
    """Count values, collapsing the tail when there are more than `limit`."""
    counts = series.fillna("(missing)").astype(str).value_counts()
    shown = counts.head(limit)
    table = pd.DataFrame(
        {
            "count": shown.astype(int),
            "pct": (shown / len(series) * 100).round(2),
        }
    )

    if len(counts) > limit:
        other = counts.iloc[limit:].sum()
        table.loc["(other)"] = [int(other), round(other / len(series) * 100, 2)]

    return table


def profile(frame: pd.DataFrame, name: str) -> None:
    """Show size, missingness, numeric summaries, and category counts."""
    print(f"{name}: {len(frame):,} rows, {frame.shape[1]} columns")
    print(f"duplicate rows: {int(frame.duplicated().sum()):,}")
    display(missingness(frame))

    numeric = numeric_summary(frame)
    if not numeric.empty:
        print("numeric summary")
        display(numeric.round(2))

    for column in frame.columns:
        series = frame[column]
        distinct = series.nunique(dropna=True)
        skip_id = column in ID_COLUMNS or column in DATE_COLUMNS
        skip_numeric = pd.api.types.is_numeric_dtype(series) and distinct > 12
        skip_unique = distinct == len(series)
        if skip_id or skip_numeric or skip_unique:
            continue

        print(column)
        display(category_counts(series))


def _check(
    name: str,
    passed: bool,
    detail: str,
) -> dict[str, object]:
    """Build one integrity-check row."""
    return {"check": name, "passed": bool(passed), "detail": detail}


def _missing_count(values: pd.Series, allowed: pd.Series) -> int:
    """Count non-null values that are absent from the allowed set."""
    return int((~values.dropna().isin(set(allowed.dropna()))).sum())


def integrity_report(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Check identifiers, foreign keys, and culture isolate counts."""
    patients = frames["patients.csv"]
    wards = frames["wards.csv"]
    antibiotics = frames["antibiotics.csv"]
    cultures = frames["cultures.csv"]
    isolates = frames["isolates.csv"]
    susceptibility = frames["susceptibility.csv"]

    counted = isolates.groupby("culture_id").size()
    compared = cultures[["culture_id", "n_isolates"]].merge(
        counted.rename("counted"),
        left_on="culture_id",
        right_index=True,
        how="left",
    )
    compared["counted"] = compared["counted"].fillna(0).astype(int)
    isolate_mismatches = int((compared["n_isolates"] != compared["counted"]).sum())

    organism_link = susceptibility.merge(
        isolates[["isolate_id", "organism"]],
        on="isolate_id",
        how="left",
        suffixes=("", "_isolate"),
    )
    organism_mismatches = int(
        (organism_link["organism"] != organism_link["organism_isolate"]).sum()
    )

    missing_culture_patients = _missing_count(
        cultures["patient_id"],
        patients["patient_id"],
    )
    missing_culture_wards = _missing_count(
        cultures["ward_code"],
        wards["ward_code"],
    )
    missing_isolate_cultures = _missing_count(
        isolates["culture_id"],
        cultures["culture_id"],
    )
    missing_susceptibility_isolates = _missing_count(
        susceptibility["isolate_id"],
        isolates["isolate_id"],
    )
    missing_susceptibility_drugs = _missing_count(
        susceptibility["antibiotic_code"],
        antibiotics["antibiotic_code"],
    )

    outpatients = cultures["ward_type"].eq("out")
    inpatients = cultures["ward_type"].eq("in")
    admission_ok = bool(
        cultures.loc[outpatients, "admission_date"].isna().all()
        and cultures.loc[inpatients, "admission_date"].notna().all()
    )

    rows = [
        _check(
            "patients.patient_id is unique",
            patients["patient_id"].is_unique,
            f"{patients['patient_id'].nunique():,} ids",
        ),
        _check(
            "wards.ward_code is unique",
            wards["ward_code"].is_unique,
            f"{len(wards):,} codes",
        ),
        _check(
            "antibiotics.antibiotic_code is unique",
            antibiotics["antibiotic_code"].is_unique,
            f"{len(antibiotics):,} codes",
        ),
        _check(
            "cultures.culture_id is unique",
            cultures["culture_id"].is_unique,
            f"{cultures['culture_id'].nunique():,} ids",
        ),
        _check(
            "isolates.isolate_id is unique",
            isolates["isolate_id"].is_unique,
            f"{isolates['isolate_id'].nunique():,} ids",
        ),
        _check(
            "cultures.patient_id exists in patients",
            missing_culture_patients == 0,
            f"{missing_culture_patients:,} missing",
        ),
        _check(
            "cultures.ward_code exists in wards",
            missing_culture_wards == 0,
            f"{missing_culture_wards:,} missing",
        ),
        _check(
            "isolates.culture_id exists in cultures",
            missing_isolate_cultures == 0,
            f"{missing_isolate_cultures:,} missing",
        ),
        _check(
            "cultures.n_isolates matches isolate rows",
            isolate_mismatches == 0,
            f"{isolate_mismatches:,} mismatches",
        ),
        _check(
            "susceptibility.isolate_id exists in isolates",
            missing_susceptibility_isolates == 0,
            f"{missing_susceptibility_isolates:,} missing",
        ),
        _check(
            "susceptibility.antibiotic_code exists in antibiotics",
            missing_susceptibility_drugs == 0,
            f"{missing_susceptibility_drugs:,} missing",
        ),
        _check(
            "susceptibility.organism matches the isolate",
            organism_mismatches == 0,
            f"{organism_mismatches:,} mismatches",
        ),
        _check(
            "admission_date is filled only for inpatients",
            admission_ok,
            "outpatients blank, inpatients filled",
        ),
    ]
    return pd.DataFrame(rows)


def phenotype_by_organism(isolates: pd.DataFrame) -> pd.DataFrame:
    """Count ESBL, MRSA, VRE, and CPE positives by organism."""
    flags = ["esbl", "mrsa", "vre", "cpe"]
    rows = []

    for organism, group in isolates.groupby("organism"):
        row = {"organism": organism, "n": len(group)}
        for flag in flags:
            row[f"{flag}_n"] = int(group[flag].sum())
            row[f"{flag}_pct"] = round(float(group[flag].mean() * 100), 2)
        rows.append(row)

    table = pd.DataFrame(rows)
    return table.sort_values("n", ascending=False).reset_index(drop=True)


def quantity_by_specimen(isolates: pd.DataFrame, cultures: pd.DataFrame) -> pd.DataFrame:
    """Count isolate quantity values within each specimen type."""
    linked = isolates.merge(
        cultures[["culture_id", "specimen"]],
        on="culture_id",
        how="left",
    )

    return pd.crosstab(
        linked["specimen"],
        linked["quantity"].fillna("(missing)"),
        margins=True,
    )


def resistance_by_antibiotic(
    susceptibility: pd.DataFrame,
    antibiotics: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize susceptible and resistant counts for each antibiotic."""
    labelled = susceptibility.assign(resistant=susceptibility["result"].eq("R"))
    counts = labelled.groupby(["antibiotic_code", "antibiotic"], as_index=False).agg(
        n=("result", "size"),
        resistant=("resistant", "sum"),
    )
    counts["resistant"] = counts["resistant"].astype(int)
    counts["susceptible"] = counts["n"] - counts["resistant"]
    counts["resistant_pct"] = (counts["resistant"] / counts["n"] * 100).round(1)

    order = antibiotics["antibiotic_code"].tolist()
    counts["antibiotic_code"] = pd.Categorical(
        counts["antibiotic_code"],
        categories=order,
        ordered=True,
    )
    return counts.sort_values("antibiotic_code").reset_index(drop=True)


def organism_antibiotic_tables(
    susceptibility: pd.DataFrame,
    isolate_ids: pd.Series,
    drug_order: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return tested counts and percent resistant by organism and antibiotic."""
    subset = susceptibility.loc[
        susceptibility["isolate_id"].isin(set(isolate_ids))
    ].copy()
    subset["resistant"] = subset["result"].eq("R")
    grouped = subset.groupby(["organism", "antibiotic_code"], as_index=False).agg(
        n=("resistant", "size"),
        resistant_n=("resistant", "sum"),
    )
    grouped["resistant_pct"] = (grouped["resistant_n"] / grouped["n"] * 100).round(1)

    counts = grouped.pivot(
        index="organism",
        columns="antibiotic_code",
        values="n",
    ).reindex(columns=drug_order)
    percent = grouped.pivot(
        index="organism",
        columns="antibiotic_code",
        values="resistant_pct",
    ).reindex(columns=drug_order)
    return counts, percent


def first_isolates(isolates: pd.DataFrame, cultures: pd.DataFrame) -> pd.DataFrame:
    """Keep the earliest isolate per patient, organism, and specimen."""
    linked = isolates.merge(
        cultures[["culture_id", "patient_id", "sample_date", "specimen"]],
        on="culture_id",
        how="left",
    )

    linked = linked.sort_values(
        ["patient_id", "organism", "specimen", "sample_date", "isolate_id"],
    )
    return linked.drop_duplicates(
        ["patient_id", "organism", "specimen"],
        keep="first",
    )


def urine_ecoli_resistance_by_sex(
    susceptibility: pd.DataFrame,
    earliest: pd.DataFrame,
    patients: pd.DataFrame,
    drug_order: list[str],
) -> pd.DataFrame:
    """Percent resistant for first urinary E. coli isolates, split by sex."""
    selected = earliest.loc[
        earliest["organism"].eq("Escherichia coli") & earliest["specimen"].eq("urine"),
        ["isolate_id", "patient_id"],
    ]
    selected = selected.merge(
        patients[["patient_id", "sex"]],
        on="patient_id",
        how="left",
    )

    rows = susceptibility.merge(
        selected,
        on="isolate_id",
        how="inner",
    )
    rows = rows.assign(resistant=rows["result"].eq("R"))
    counts = rows.groupby(["antibiotic_code", "sex"], as_index=False).agg(
        n=("resistant", "size"),
        resistant_n=("resistant", "sum"),
    )
    counts["resistant_pct"] = (counts["resistant_n"] / counts["n"] * 100).round(1)

    percent = counts.pivot(
        index="antibiotic_code",
        columns="sex",
        values="resistant_pct",
    ).reindex(drug_order)
    tested = counts.pivot(
        index="antibiotic_code",
        columns="sex",
        values="n",
    ).reindex(drug_order)
    combined = percent.add_suffix("_pct").join(tested.add_suffix("_n"))
    column_order = [
        column
        for column in ["F_pct", "M_pct", "F_n", "M_n"]
        if column in combined.columns
    ]
    return combined.loc[:, column_order].dropna(how="all")


: 

## Overview

Row counts, missing cells, duplicate rows, and identifier checks across the six files.


In [ ]:
frames = load_tables(DATA_DIR)

overview = pd.DataFrame(
    [
        {
            "file": name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
            "memory_mb": round(frame.memory_usage(deep=True).sum() / 1_000_000, 2),
        }
        for name, frame in frames.items()
    ]
)
display(overview)
display(integrity_report(frames))


## patients.csv

One row per patient: sex, birth year, and home area.


In [ ]:
patients = frames["patients.csv"]
profile(patients, "patients.csv")

cultured = frames["cultures.csv"]["patient_id"].nunique()
print(f"patients with at least one culture: {cultured:,} of {len(patients):,}")

birth_decade = (patients["birth_year"] // 10 * 10).astype(str) + "s"
print("birth decade")
display(category_counts(birth_decade).sort_index())


## wards.csv

Ward code list. `ward_type` is `in` or `out`. `institution` is `hospital` or `primary_care`.


In [ ]:
wards = frames["wards.csv"]
profile(wards, "wards.csv")
display(wards)


## antibiotics.csv

Antibiotic code list. `reported_for` is `all` or `urine`. Urinary agents are reported for urine cultures.


In [ ]:
antibiotics = frames["antibiotics.csv"]
profile(antibiotics, "antibiotics.csv")
display(antibiotics)


## cultures.csv

One row per culture. `result` is `positive`, `negative`, or `mixed_growth`. Inpatients have an admission date. Outpatient admission dates are blank.

Hospital day 1 is the admission date. A sample on hospital day 3 or later is the usual proxy for a hospital-onset finding.


In [ ]:
cultures = frames["cultures.csv"]
profile(cultures, "cultures.csv")

print(
    f"sample dates: {cultures['sample_date'].min().date()} "
    f"to {cultures['sample_date'].max().date()}"
)

print("cultures per ISO week")
weekly = cultures.groupby(["iso_year", "iso_week"]).size()
display(weekly.describe().round(1).to_frame("cultures"))

print("cultures per patient, among patients who have a culture")
per_patient = cultures.groupby("patient_id").size()
display(per_patient.describe().round(2).to_frame("cultures"))

age_bands = pd.cut(
    cultures["age_years"],
    bins=[0, 18, 65, 80, 100],
    labels=["0–17", "18–64", "65–79", "80–99"],
    right=False,
)
print("age band")
band_counts = age_bands.value_counts(sort=False).rename("count").to_frame()
band_counts["pct"] = (band_counts["count"] / band_counts["count"].sum() * 100).round(1)
display(band_counts)

print("age by specimen")
display(cultures.groupby("specimen")["age_years"].describe().round(1))

print("n_isolates by result")
display(
    pd.crosstab(
        cultures["result"],
        cultures["n_isolates"],
        margins=True,
    )
)

print("result by specimen (counts)")
display(
    pd.crosstab(
        cultures["specimen"],
        cultures["result"],
        margins=True,
    )
)
print("result by specimen (row %)")
display(
    pd.crosstab(
        cultures["specimen"],
        cultures["result"],
        normalize="index",
    ).mul(100).round(1)
)

print("result by ward type (row %)")
display(
    pd.crosstab(
        cultures["ward_type"],
        cultures["result"],
        normalize="index",
    ).mul(100).round(1)
)

print("specimen by ward")
display(
    pd.crosstab(
        cultures["ward_code"],
        cultures["specimen"],
        margins=True,
    )
)

inpatients = cultures.loc[cultures["ward_type"].eq("in")].copy()
inpatients["hospital_day"] = (
    (inpatients["sample_date"] - inpatients["admission_date"]).dt.days + 1
)
onset = inpatients["hospital_day"].ge(3)
print(
    f"inpatient cultures: {len(inpatients):,}; "
    f"hospital day 3 or later: {int(onset.sum()):,} ({onset.mean() * 100:.1f}%)"
)
print(
    "admission dates: "
    f"{inpatients['admission_date'].min().date()} "
    f"to {inpatients['admission_date'].max().date()}"
)
print("hospital day (1 = admission date)")
display(inpatients["hospital_day"].describe().round(1).to_frame("hospital_day"))


## isolates.csv

One row per isolate. `quantity` is a urine CFU/ml band or a wound score (`+` to `+++`). Blood isolates have no quantity. Phenotype flags are 0/1.


In [ ]:
isolates = frames["isolates.csv"]
profile(isolates, "isolates.csv")

phenotype_columns = ["esbl", "mrsa", "vre", "cpe"]
print("phenotype positives")
display(
    pd.DataFrame(
        {
            "positive": isolates[phenotype_columns].sum().astype(int),
            "pct": (isolates[phenotype_columns].mean() * 100).round(2),
        }
    )
)

print("phenotypes by organism")
display(phenotype_by_organism(isolates))

print("quantity by specimen")
display(quantity_by_specimen(isolates, cultures))


## susceptibility.csv

One row per isolate and antibiotic. `result` is `S` or `R`.

Percent resistant is the share of `R` results. The all-rows tables count every result in this file. The first-isolate tables keep the earliest isolate per patient, organism, and specimen over the whole date range, and leave a percentage blank when fewer than 30 isolates were tested. `<NA>` is an organism–antibiotic pair that is not reported. Urinary *E. coli* is also split by sex.


In [ ]:
susceptibility = frames["susceptibility.csv"]
profile(susceptibility, "susceptibility.csv")

resistant_n = int(susceptibility["result"].eq("R").sum())
print(
    f"resistant rows: {resistant_n:,} of {len(susceptibility):,} "
    f"({resistant_n / len(susceptibility) * 100:.1f}%)"
)

print("results per isolate")
per_isolate = susceptibility.groupby("isolate_id").size()
display(per_isolate.describe().round(2).to_frame("results"))

print("resistance by antibiotic (all rows)")
display(resistance_by_antibiotic(susceptibility, antibiotics))

drug_order = antibiotics["antibiotic_code"].tolist()
organism_order = isolates["organism"].value_counts().index.tolist()
counts, percent = organism_antibiotic_tables(
    susceptibility,
    susceptibility["isolate_id"],
    drug_order,
)
counts = counts.reindex(organism_order).astype("Int64")
percent = percent.reindex(organism_order)
print("isolates tested, by organism and antibiotic")
display(counts)
print("percent resistant, by organism and antibiotic (all rows)")
display(percent)

earliest = first_isolates(isolates, cultures)
print(
    f"first isolates (one per patient, organism, and specimen): "
    f"{len(earliest):,} of {len(isolates):,}"
)
first_counts, first_percent = organism_antibiotic_tables(
    susceptibility,
    earliest["isolate_id"],
    drug_order,
)
first_counts = first_counts.reindex(organism_order).astype("Int64")
first_percent = first_percent.reindex(organism_order).where(first_counts.ge(30))
print("first isolates tested, by organism and antibiotic")
display(first_counts)
print("first-isolate percent resistant; blank when fewer than 30 isolates were tested")
display(first_percent)

urine_first = earliest.loc[
    earliest["organism"].eq("Escherichia coli") & earliest["specimen"].eq("urine")
]
print(f"first urinary E. coli isolates: {len(urine_first):,}")
print("first urinary E. coli, percent resistant by sex")
display(
    urine_ecoli_resistance_by_sex(
        susceptibility,
        earliest,
        patients,
        drug_order,
    )
)
